In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, round, datediff, when, to_timestamp

spark = SparkSession.builder.appName("olist_gold").getOrCreate()

df_orders = spark.read.parquet('../data/silver/orders')
df_order_items = spark.read.parquet('../data/silver/order_items')
df_order_payments = spark.read.parquet('../data/silver/order_payments')
df_order_reviews = spark.read.parquet('../data/silver/order_reviews')
df_customers = spark.read.parquet('../data/silver/customers')
df_products = spark.read.parquet('../data/silver/products')
df_sellers = spark.read.parquet('../data/silver/sellers')
df_category = spark.read.parquet('../data/silver/category')
df_geolocation = spark.read.parquet('../data/silver/geolocation')

In [ ]:
df_gold = df_orders \
    .join(df_customers, "customer_id") \
    .join(df_order_items, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left") \
    #.join(df_order_reviews, "order_id", "left")
    #.join(df_order_payments, "order_id", "left") \
    


In [ ]:
print(f"Nombre de lignes : {df_gold.count()}")
print(f"Nombre de colonnes : {len(df_gold.columns)}")

cols_importantes = ["order_id", "customer_id", "product_id", "seller_id", "price"]
for c in cols_importantes:
    nulls = df_gold.filter(col(c).isNull()).count()
    print(f"  {c} : {nulls} nulls")

df_gold.select("order_id", "customer_id", "price", "product_category_name_english").show(5)

In [ ]:
from pyspark.sql.functions import sum, avg, round, count, desc, month, year

df_items_base = df_order_items \
    .join(df_orders, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left")

# Chiffre d'affaires total
df_items_base.agg(round(sum("price"), 2).alias("ca_total")).show()

# Panier moyen par commande
df_items_base.groupBy("order_id").agg(sum("price").alias("total")) \
    .agg(round(avg("total"), 2).alias("panier_moyen")).show()

# Top 10 catégories
df_items_base.groupBy("product_category_name_english") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

# Top 10 vendeurs
df_items_base.groupBy("seller_id") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

In [ ]:
from pyspark.sql.functions import month, year, trunc

df_items_base \
    .withColumn("mois", trunc("order_purchase_timestamp", "MM")) \
    .groupBy("mois") \
    .agg(round(sum("price"), 2).alias("ca_mensuel")) \
    .orderBy("mois") \
    .show(24)

In [ ]:
# Délai moyen de livraison et taux de retard
from pyspark.sql.functions import datediff, when

df_livraison = df_orders.filter(col("order_delivered_customer_date").isNotNull())

# Délai moyen de livraison
df_livraison \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .show()

# Taux de retard
df_livraison \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .agg(
        round(avg("en_retard") * 100, 2).alias("taux_retard_pct")
    ) \
    .show()

In [ ]:
# Note moyenne client
df_order_reviews.agg(round(avg("review_score"), 2).alias("note_moyenne")).show()

# Impact des retards sur la satisfaction
df_orders \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "retard"
    ).otherwise("a_temps")) \
    .join(df_order_reviews, "order_id", "left") \
    .groupBy("en_retard") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

In [ ]:
# Taux de commandes par statut
df_orders \
    .groupBy("order_status") \
    .agg(count("order_id").alias("nb_commandes")) \
    .orderBy(desc("nb_commandes")) \
    .show()

# CA par état brésilien
df_items_base \
    .join(df_customers, "customer_id") \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

# Note moyenne par catégorie
df_items_base \
    .join(df_order_reviews, "order_id", "left") \
    .groupBy("product_category_name_english") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .orderBy(desc("note_moyenne")) \
    .show(10)

In [ ]:
df_items_base \
    .join(df_customers, "customer_id") \
    .join(df_geolocation, 
          df_customers.customer_zip_code_prefix == df_geolocation.geolocation_zip_code_prefix,
          "left") \
    .groupBy("customer_state", "geolocation_lat", "geolocation_lng") \
    .agg(
        round(sum("price"), 2).alias("ca"),
        count("order_id").alias("nb_commandes")
    ) \
    .orderBy(desc("ca")) \
    .show(10)

In [ ]:
df_items_base \
    .join(df_customers.select("customer_id", "customer_state"), "customer_id") \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)